# Extracción de opiniones Trustpilot — Lululemon vs ALO Yoga

Scraping de reviews de Trustpilot para dos marcas:
- **Lululemon**: https://es.trustpilot.com/review/www.lululemon.com
- **ALO Yoga**: https://es.trustpilot.com/review/www.aloyoga.com

Se extraen todas las opiniones con el país del reviewer (`consumer.countryCode`).

**Método:** Se parsea el JSON embebido en `__NEXT_DATA__` de cada página (no hace falta API externa).

In [1]:
pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd
import json
import re
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

---
## 1. Función de scraping

Cada página de Trustpilot tiene un `<script id="__NEXT_DATA__">` con un JSON que contiene:
- `props.pageProps.reviews` — lista de 20 reviews por página
- `props.pageProps.filters.pagination` — info de paginación

Cada review tiene:
- `text` — texto de la opinión
- `title` — título
- `rating` — 1 a 5
- `language` — idioma (en, es, fr, de...)
- `consumer.displayName` — nombre del reviewer
- `consumer.countryCode` — país (US, GB, ES, FR, DE...)
- `consumer.numberOfReviews` — número de opiniones del usuario
- `dates.publishedDate` — fecha de publicación
- `dates.experiencedDate` — fecha de la experiencia
- `labels.verification.isVerified` — si está verificada
- `source` — origen (Organic, Invitation...)

In [3]:
def scrape_trustpilot(company_slug, max_pages=None):
    """
    Extrae todas las reviews de una empresa en Trustpilot.
    
    Args:
        company_slug: parte de la URL (ej: 'www.lululemon.com')
        max_pages: límite de páginas (None = todas)
    
    Returns:
        Lista de dicts con las reviews
    """
    base_url = f"https://es.trustpilot.com/review/{company_slug}"
    all_reviews = []
    page = 1
    total_pages = None

    # Primero leer página 2 para obtener la paginación real
    # (la página 1 con languages=all devuelve totalPages=1 por un bug de Trustpilot)
    print("  Leyendo paginación real...", end=" ")
    resp_check = requests.get(f"{base_url}?languages=all&page=2", headers=HEADERS)
    if resp_check.status_code == 200:
        match_check = re.search(r'<script[^>]*id="__NEXT_DATA__"[^>]*>(.*?)</script>', resp_check.text, re.DOTALL)
        if match_check:
            data_check = json.loads(match_check.group(1))
            pag_check = data_check["props"]["pageProps"].get("filters", {}).get("pagination", {})
            total_pages = pag_check.get("totalPages", 1)
            total_count = pag_check.get("totalCount", 0)
            print(f"Total: {total_count} reviews en {total_pages} páginas")

    if total_pages is None:
        total_pages = 1
        print("no se pudo leer, se intentará página a página")

    while page <= total_pages:
        url = f"{base_url}?languages=all&page={page}"
        print(f"  Página {page}/{total_pages}...", end=" ")

        resp = requests.get(url, headers=HEADERS)
        if resp.status_code != 200:
            print(f"ERROR: status {resp.status_code}")
            break

        # Extraer __NEXT_DATA__
        match = re.search(r'<script[^>]*id="__NEXT_DATA__"[^>]*>(.*?)</script>', resp.text, re.DOTALL)
        if not match:
            print("ERROR: no se encontró __NEXT_DATA__")
            break

        data = json.loads(match.group(1))
        page_props = data["props"]["pageProps"]
        reviews = page_props.get("reviews", [])

        if not reviews:
            print("sin reviews, fin.")
            break

        # Extraer campos relevantes de cada review
        for r in reviews:
            consumer = r.get("consumer", {})
            dates = r.get("dates", {})
            verification = r.get("labels", {}).get("verification", {})

            all_reviews.append({
                "review_id": r.get("id"),
                "title": r.get("title"),
                "text": r.get("text"),
                "rating": r.get("rating"),
                "language": r.get("language"),
                "reviewer_name": consumer.get("displayName"),
                "reviewer_country": consumer.get("countryCode"),
                "reviewer_num_reviews": consumer.get("numberOfReviews"),
                "is_verified": verification.get("isVerified", False),
                "source": r.get("source"),
                "published_date": dates.get("publishedDate"),
                "experience_date": dates.get("experiencedDate"),
            })

        print(f"{len(reviews)} reviews (total: {len(all_reviews)})")

        if max_pages and page >= max_pages:
            print(f"  Límite de {max_pages} páginas alcanzado.")
            break

        page += 1
        time.sleep(1)  # pausa para no saturar

    print(f"\nTotal extraídas: {len(all_reviews)} reviews")
    return all_reviews

print("Función de scraping cargada.")

Función de scraping cargada.


---
## 2. Scraping — Lululemon

In [4]:
print("Extrayendo reviews de Lululemon...\n")
reviews_lululemon = scrape_trustpilot("www.lululemon.com")

Extrayendo reviews de Lululemon...

  Leyendo paginación real... Total: 1381 reviews en 70 páginas
  Página 1/70... 8 reviews (total: 8)
  Página 2/70... 20 reviews (total: 28)
  Página 3/70... 20 reviews (total: 48)
  Página 4/70... 20 reviews (total: 68)
  Página 5/70... 20 reviews (total: 88)
  Página 6/70... 20 reviews (total: 108)
  Página 7/70... 20 reviews (total: 128)
  Página 8/70... 20 reviews (total: 148)
  Página 9/70... 20 reviews (total: 168)
  Página 10/70... 20 reviews (total: 188)
  Página 11/70... sin reviews, fin.

Total extraídas: 188 reviews


In [5]:
df_lulu = pd.DataFrame(reviews_lululemon)

print(f"Total reviews Lululemon: {len(df_lulu)}")
print(f"\nDistribución por rating:")
print(df_lulu["rating"].value_counts().sort_index())
print(f"\nTop 10 países:")
print(df_lulu["reviewer_country"].value_counts().head(10))
print(f"\nIdiomas:")
print(df_lulu["language"].value_counts().head(10))

df_lulu.head(5)

Total reviews Lululemon: 188

Distribución por rating:
rating
1    150
2     10
3      5
4      7
5     16
Name: count, dtype: int64

Top 10 países:
reviewer_country
US    69
CA    45
GB    16
ES    12
FR     8
DE     6
NL     4
AU     4
IE     4
DK     2
Name: count, dtype: int64

Idiomas:
language
en    153
es     12
fr      9
de      7
nl      3
sv      2
da      1
ja      1
Name: count, dtype: int64


,review_id,title,text,rating,language,reviewer_name,reviewer_country,reviewer_num_reviews,is_verified,source,published_date,experience_date
0,698f5a52b1c96308944c4701,El 8 de enero hice un pedido por valor…,El 8 de enero hice un pedido por valor de 500€...,1,es,CAROLINA,ES,4,False,Organic,2026-02-13T19:07:30.000Z,2026-01-08T00:00:00.000Z
1,695cedaa651313c9714ed6e7,Hice un pedido el 25 de diciembre,"Hice un pedido el 25 de diciembre, me dijeron ...",1,es,Johanna,ES,2,False,Organic,2026-01-06T13:10:34.000Z,2025-12-25T00:00:00.000Z
2,695455e30f3d08e92102e95d,BUENAS TARDES,"BUENAS TARDES, \nNO TENGO UNA OPINION, \nSI TE...",5,es,fabulous Sundays,ES,1,False,Organic,2025-12-31T00:44:51.000Z,2025-12-30T00:00:00.000Z
3,6939419888249807ec8ea87c,Estafa,He sufrido una estafa donde han utilizado mi t...,1,es,Patricia AC,ES,3,False,Organic,2025-12-10T11:47:04.000Z,2025-11-12T00:00:00.000Z
4,679a2ac510e80efc85a9f337,Desastroso como no he experimentado en mi vida…,Desastroso como no he experimentado en mi vida...,1,es,rarafriend2,ES,18,False,Organic,2025-01-29T15:19:01.000Z,2025-01-09T00:00:00.000Z


In [6]:
# Guardar
df_lulu.to_csv("../datos/processed/trustpilot/trustpilot_lululemon.csv", index=False, encoding="utf-8-sig")

with open("../datos/raw/trustpilot/trustpilot_lululemon.json", "w", encoding="utf-8") as f:
    json.dump(reviews_lululemon, f, ensure_ascii=False, indent=4)

print("Guardado: ../datos/processed/trustpilot/trustpilot_lululemon.csv / .json")

Guardado: ../datos/processed/trustpilot/trustpilot_lululemon.csv / .json


---
## 3. Scraping — ALO Yoga

In [7]:
print("Extrayendo reviews de ALO Yoga...\n")
reviews_alo = scrape_trustpilot("www.aloyoga.com")

Extrayendo reviews de ALO Yoga...

  Leyendo paginación real... Total: 1118 reviews en 56 páginas
  Página 1/56... 15 reviews (total: 15)
  Página 2/56... 20 reviews (total: 35)
  Página 3/56... 20 reviews (total: 55)
  Página 4/56... 20 reviews (total: 75)
  Página 5/56... 20 reviews (total: 95)
  Página 6/56... 20 reviews (total: 115)
  Página 7/56... 20 reviews (total: 135)
  Página 8/56... 20 reviews (total: 155)
  Página 9/56... 20 reviews (total: 175)
  Página 10/56... 20 reviews (total: 195)
  Página 11/56... sin reviews, fin.

Total extraídas: 195 reviews


In [8]:
df_alo = pd.DataFrame(reviews_alo)

print(f"Total reviews ALO Yoga: {len(df_alo)}")
print(f"\nDistribución por rating:")
print(df_alo["rating"].value_counts().sort_index())
print(f"\nTop 10 países:")
print(df_alo["reviewer_country"].value_counts().head(10))
print(f"\nIdiomas:")
print(df_alo["language"].value_counts().head(10))

df_alo.head(5)

Total reviews ALO Yoga: 195

Distribución por rating:
rating
1    175
2      2
3      8
4      1
5      9
Name: count, dtype: int64

Top 10 países:
reviewer_country
US    63
GB    53
ES    13
CA    13
DE    10
IE     7
NL     7
IT     4
AU     4
FR     4
Name: count, dtype: int64

Idiomas:
language
en    165
es     16
de      7
fr      3
nl      2
it      2
Name: count, dtype: int64


,review_id,title,text,rating,language,reviewer_name,reviewer_country,reviewer_num_reviews,is_verified,source,published_date,experience_date
0,697672ebad37e116f27cf335,Muy buena calidad,Muy buena calidad,5,es,Santiago Torres,MX,1,False,Organic,2026-01-25T21:45:47.000Z,2026-01-08T00:00:00.000Z
1,68effa97f80fe698e86e03df,Realice una devolución desde España y…,Realice una devolución desde España y fue impo...,1,es,Vanessa Chacin Ayala,ES,1,False,Organic,2025-10-15T21:48:39.000Z,2025-10-01T00:00:00.000Z
2,68e4da55520db7e3ef56f161,Malísima calidad no he ni lavado por…,Malísima calidad no he ni lavado por primera v...,1,es,Vanessa Chacin,ES,1,False,Organic,2025-10-07T11:16:05.000Z,2025-10-07T00:00:00.000Z
3,67a21d53ac7f75ba733698ba,No devuelven tu dinero,He hecho ya dos devoluciones y no me han envia...,1,es,Leyla Mejia,IT,12,False,Organic,2025-02-04T15:59:47.000Z,2025-02-04T00:00:00.000Z
4,6795279f4f99482175644e9c,¡MENUDO TIMO,¡MENUDO TIMO! Lamento mucho no haber leído los...,1,es,J León,ES,2,False,Organic,2025-01-25T20:04:15.000Z,2024-12-03T00:00:00.000Z


In [9]:
# Guardar
df_alo.to_csv("../datos/processed/trustpilot/trustpilot_alo_yoga.csv", index=False, encoding="utf-8-sig")

with open("../datos/raw/trustpilot/trustpilot_alo_yoga.json", "w", encoding="utf-8") as f:
    json.dump(reviews_alo, f, ensure_ascii=False, indent=4)

print("Guardado: ../datos/processed/trustpilot/trustpilot_alo_yoga.csv / .json")

Guardado: ../datos/processed/trustpilot/trustpilot_alo_yoga.csv / .json


---
## 4. Resumen comparativo

In [10]:
print("="*60)
print("RESUMEN COMPARATIVO")
print("="*60)

for nombre, df in [("Lululemon", df_lulu), ("ALO Yoga", df_alo)]:
    print(f"\n--- {nombre} ---")
    print(f"  Total reviews: {len(df)}")
    print(f"  Rating medio: {df['rating'].mean():.2f}")
    print(f"  % 1 estrella: {(df['rating'] == 1).mean()*100:.1f}%")
    print(f"  % 5 estrellas: {(df['rating'] == 5).mean()*100:.1f}%")
    print(f"  Países únicos: {df['reviewer_country'].nunique()}")
    print(f"  Top 5 países: {df['reviewer_country'].value_counts().head(5).to_dict()}")
    print(f"  Reviews verificadas: {df['is_verified'].sum()} ({df['is_verified'].mean()*100:.1f}%)")

print(f"\nArchivos generados:")
print(f"  ../datos/processed/trustpilot/trustpilot_lululemon.csv / .json")
print(f"  ../datos/processed/trustpilot/trustpilot_alo_yoga.csv / .json")

RESUMEN COMPARATIVO

--- Lululemon ---
  Total reviews: 188
  Rating medio: 1.56
  % 1 estrella: 79.8%
  % 5 estrellas: 8.5%
  Países únicos: 26
  Top 5 países: {'US': 69, 'CA': 45, 'GB': 16, 'ES': 12, 'FR': 8}
  Reviews verificadas: 0 (0.0%)

--- ALO Yoga ---
  Total reviews: 195
  Rating medio: 1.29
  % 1 estrella: 89.7%
  % 5 estrellas: 4.6%
  Países únicos: 24
  Top 5 países: {'US': 63, 'GB': 53, 'ES': 13, 'CA': 13, 'DE': 10}
  Reviews verificadas: 0 (0.0%)

Archivos generados:
  ../datos/processed/trustpilot/trustpilot_lululemon.csv / .json
  ../datos/processed/trustpilot/trustpilot_alo_yoga.csv / .json
